# **Voyage Analytics: Personalized Hotel Recommendation System**


##### **Project Type** - Recommendation System / MLOps  
##### **Contribution** - Individual  
##### **Team Member 1 - Tarun Sreekar Parasa**


# **Project Summary -**

This notebook develops the hotel recommendation component of the Voyage Analytics capstone. The hotels dataset contains 40,552 hotel-booking records from 1,310 users, covering nine hotels in nine cities. Each row includes the travel code, user code, hotel name, destination, number of stay days, daily price, total booking value and booking date. The recommendation objective is to suggest hotels that are most relevant to a user based on historical booking behavior while providing a sensible popularity fallback for users with limited history.

Exploratory analysis shows that the recommendation problem is compact and unusually dense. There are only nine hotels, and the average active user has interacted with a large share of the available catalogue. A complex deep-learning recommender or large matrix-factorization model would therefore be unnecessary and difficult to justify. Instead, the notebook compares three transparent ranking strategies: a global popularity recommender, a personalized booking-frequency recommender, and a hybrid recommender combining user history, item similarity, price preference and global popularity.

Evaluation uses a temporal holdout. For each user with at least two bookings, the latest booking is removed from the training interactions and used as the test target. The recommender then ranks hotels using only the user's earlier history. This setup better reflects the real task of predicting a later choice from past behavior. Performance is measured using Hit Rate@K and Mean Reciprocal Rank@K. In the validated benchmark, the global popularity model achieved Hit Rate@3 of approximately 0.353, the personalized frequency model achieved approximately 0.391, and the tested hybrid achieved approximately 0.366. The simpler personalized frequency approach therefore performs best on this particular dense nine-item catalogue.

The final recommender scores each hotel primarily from the user's normalized historical booking frequency and adds a small popularity prior to break ties and support sparse histories. For new users with no booking history, the system falls back to globally popular hotels. The fitted recommender state—including hotel catalogue, popularity scores, user-item interaction matrix and hotel metadata—is saved with Joblib so it can be loaded by the Streamlit application.

The resulting model is intentionally explainable. Each recommendation can be accompanied by a reason such as “frequently booked by this user” or “popular among travellers.” This is valuable for a small catalogue, where transparency and operational simplicity are more useful than unnecessary model complexity. The Streamlit application built later in the project will use this recommender to display user travel history, hotel suggestions and supporting travel insights.


# **GitHub Link -**

**GitHub Repository:** [https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel](https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel)


# **Problem Statement**


Build and evaluate a hotel recommendation model that ranks hotels for each user from historical booking behavior. Use a temporal holdout to test whether past bookings can recover a later hotel choice, support cold-start users with a popularity fallback, and persist the recommender state for a Streamlit application.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required. 
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits. 
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule. 

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE=42


### Dataset Loading

In [ ]:
# Load Dataset
possible_paths=[Path('data/raw/hotels.csv'),Path('/content/hotels.csv'),Path('/content/drive/MyDrive/Voyage-Analytics/hotels.csv'),Path('/mnt/data/voyage_work/hotels.csv')]
DATA_PATH=next((p for p in possible_paths if p.exists()),None)
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded=files.upload(); DATA_PATH=Path(next(iter(uploaded.keys())))
    except Exception as exc: raise FileNotFoundError('Could not locate hotels.csv') from exc
hotels_df=pd.read_csv(DATA_PATH); print(f'Loaded dataset from: {DATA_PATH}')


### Dataset First View

In [ ]:
display(hotels_df.head())


### Dataset Rows & Columns count

In [ ]:
print(f'Rows: {hotels_df.shape[0]:,} | Columns: {hotels_df.shape[1]}')


### Dataset Information

In [ ]:
hotels_df.info()


#### Duplicate Values

In [ ]:
print('Duplicate rows:',hotels_df.duplicated().sum())


#### Missing Values/Null Values

In [ ]:
display(hotels_df.isna().sum().to_frame('missing_count'))


In [ ]:
sns.heatmap(hotels_df.isna(),cbar=False,yticklabels=False); plt.title('Missing Values'); plt.show()


### What did you know about your dataset?

The hotel dataset contains 40,552 complete booking rows and eight columns, with no missing values or exact duplicate rows. It covers 1,310 users and nine hotels. The catalogue is small, so evaluation and explainability are more important than using a large-scale recommender architecture.


## ***2. Understanding Your Variables***

In [ ]:
hotels_df.columns.tolist()


In [ ]:
display(hotels_df.describe(include='all').T)


### Variables Description 

- **travelCode:** Journey identifier.  
- **userCode:** User identifier used to personalize recommendations.  
- **name:** Hotel item to recommend.  
- **place:** Hotel destination/city.  
- **days:** Length of stay.  
- **price:** Daily hotel price.  
- **total:** Total stay price.  
- **date:** Booking date used for temporal evaluation.


### Check Unique Values for each variable.

In [ ]:
for c in hotels_df.columns:
 print(c,hotels_df[c].nunique())
 if hotels_df[c].nunique()<=15: print(hotels_df[c].value_counts().to_dict())


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
hotels=hotels_df.copy(); hotels['date']=pd.to_datetime(hotels['date'],format='%m/%d/%Y',errors='raise'); hotels['year']=hotels.date.dt.year; hotels['month']=hotels.date.dt.month; hotels['price_per_day_check']=hotels['total']/hotels['days']; user_item_counts=hotels.groupby(['userCode','name']).size().unstack(fill_value=0); print('Users:',hotels.userCode.nunique(),'Hotels:',hotels.name.nunique()); print('Average distinct hotels per user:',(user_item_counts>0).sum(axis=1).mean())


### What all manipulations have you done and insights you found?

Dates were converted to datetime, calendar fields were derived, and a user-item interaction matrix was created from booking counts. The average user has interacted with much of the nine-hotel catalogue, confirming that this is a dense small-catalogue recommendation problem.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
hotels['name'].value_counts().plot(kind='bar',figsize=(11,5)); plt.title('Bookings by Hotel'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 2

In [ ]:
hotels['place'].value_counts().plot(kind='bar',figsize=(11,5)); plt.title('Bookings by Destination'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 3

In [ ]:
sns.histplot(hotels['days'],bins=15); plt.title('Length of Stay Distribution'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 4

In [ ]:
sns.histplot(hotels['price'],bins=20); plt.title('Daily Hotel Price Distribution'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 5

In [ ]:
sns.histplot(hotels['total'],bins=30); plt.title('Total Booking Value Distribution'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 6

In [ ]:
hotels.groupby('name')['price'].median().sort_values().plot(kind='bar',figsize=(11,5)); plt.title('Median Daily Price by Hotel'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 7

In [ ]:
hotels.groupby('name')['days'].mean().sort_values().plot(kind='bar',figsize=(11,5)); plt.title('Average Stay Length by Hotel'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 8

In [ ]:
hotels.groupby('place')['total'].mean().sort_values().plot(kind='bar',figsize=(11,5)); plt.title('Average Booking Value by Destination'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 9

In [ ]:
hotels.set_index('date').resample('M').size().plot(figsize=(12,5)); plt.title('Monthly Hotel Booking Volume'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 10

In [ ]:
hotels.groupby('year').size().plot(kind='bar'); plt.title('Bookings by Year'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 11

In [ ]:
sns.scatterplot(data=hotels.sample(min(15000,len(hotels)),random_state=RANDOM_STATE),x='days',y='total',hue='name',alpha=.3,legend=False); plt.title('Stay Length versus Total Cost'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 12

In [ ]:
(user_item_counts>0).sum(axis=1).value_counts().sort_index().plot(kind='bar'); plt.title('Distinct Hotels Visited per User'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 13

In [ ]:
sns.heatmap((user_item_counts>0).astype(int).sample(min(100,len(user_item_counts)),random_state=RANDOM_STATE),cbar=False,yticklabels=False); plt.title('Sample User-Hotel Interaction Matrix'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business implication is that simple personalized history and popularity signals can be effective and easier to explain than unnecessarily complex recommendation architectures.


#### Chart - 14 - Correlation Heatmap

In [ ]:
sns.heatmap(hotels[['days','price','total','year','month']].corr(),annot=True,cmap='coolwarm',center=0); plt.title('Correlation Heatmap'); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


#### Chart - 15 - Pair Plot 

In [ ]:
sns.pairplot(hotels[['days','price','total']].sample(3000,random_state=RANDOM_STATE),corner=True); plt.show()


##### 1. Why did you pick the specific chart?

This chart was selected to describe demand, price, stay behavior or interaction density before choosing a recommendation strategy.


##### 2. What is/are the insight(s) found from the chart?

The visualization confirms a small nine-item catalogue with repeated user interactions, meaningful differences in hotel price and booking volume, and a dense user-item matrix.


## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three hypotheses are tested: hotel booking frequencies are uniformly distributed; daily hotel prices are equal across hotels; and stay duration is associated with total booking value.


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** All hotels have equal booking probability.  
**H₁:** Booking frequencies differ across hotels.


#### 2. Perform an appropriate statistical test.

In [ ]:
counts=hotels['name'].value_counts(); chi2,p=stats.chisquare(counts.values); print(chi2,p)


##### Which statistical test have you done to obtain P-Value?

Chi-square goodness-of-fit test.


##### Why did you choose the specific statistical test?

It tests whether observed categorical booking counts differ from an equal-frequency expectation.


### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Daily price distributions are the same across hotels.  
**H₁:** At least one hotel differs.


#### 2. Perform an appropriate statistical test.

In [ ]:
groups=[g.price.values for _,g in hotels.groupby('name')]; stat,p=stats.kruskal(*groups); print(stat,p)


##### Which statistical test have you done to obtain P-Value?

Kruskal-Wallis H-test.


##### Why did you choose the specific statistical test?

It compares a continuous variable across more than two independent hotel groups without a normality assumption.


### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Stay duration and total booking value have no monotonic association.  
**H₁:** They are associated.


#### 2. Perform an appropriate statistical test.

In [ ]:
rho,p=stats.spearmanr(hotels['days'],hotels['total']); print(rho,p)


##### Which statistical test have you done to obtain P-Value?

Spearman rank-correlation test.


##### Why did you choose the specific statistical test?

The relationship is expected to be monotonic and does not require normality.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
print(hotels.isna().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

No missing-value imputation is required.


### 2. Handling Outliers

In [ ]:
print(hotels[['days','price','total']].quantile([.01,.99]))


##### What all outlier treatment techniques have you used and why did you use those techniques?

No valid bookings are removed solely as statistical outliers because long stays and high-value bookings may be genuine user preferences.


### 3. Categorical Encoding

In [ ]:
print('Recommendation modeling uses interaction counts rather than categorical one-hot features.')


#### What all categorical encoding techniques have you used & why did you use those techniques?

Hotel names become item columns in the user-item interaction matrix. No conventional supervised categorical encoder is required for the final ranking model.


### 4. Textual Data Preprocessing 
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 2. Lower Casing

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 3. Removing Punctuations

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
print('Not applicable to this structured recommendation dataset.')


In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 7. Tokenization

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 8. Text Normalization

In [ ]:
print('Not applicable to this structured recommendation dataset.')


##### Which text normalization technique have you used and why?

Text normalization and text vectorization are not required because hotel identity is represented directly as an item in the interaction matrix.


#### 9. Part of speech tagging

In [ ]:
print('Not applicable to this structured recommendation dataset.')


#### 10. Text Vectorization

In [ ]:
print('Not applicable to this structured recommendation dataset.')


##### Which text vectorization technique have you used and why?

Text normalization and text vectorization are not required because hotel identity is represented directly as an item in the interaction matrix.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature manipulation: build booking-count interactions
interaction_matrix=hotels.groupby(['userCode','name']).size().unstack(fill_value=0)
hotel_metadata=hotels.groupby('name').agg(place=('place','first'),price=('price','median'),bookings=('name','size')).sort_index(); display(hotel_metadata)


#### 2. Feature Selection

In [ ]:
print('Recommendation signals: user booking frequency, item popularity, optional similarity and price preference.')


##### What all feature selection methods have you used  and why?

The recommendation features are derived from historical interactions. IDs are meaningful here because `userCode` indexes a user profile and hotel `name` indexes the item catalogue.


##### Which all features you found important and why?

Historical booking frequency is the strongest personalized signal in this dense small catalogue. Popularity provides a robust fallback for new or sparse users.


### 5. Data Transformation

No target transformation is applicable.


In [ ]:
print('No target transformation is used; the task is ranking, not numerical prediction.')


### 6. Data Scaling

In [ ]:
print('Interaction scores are normalized during ranking.')


Per-user frequency scores are normalized to the user maximum, and popularity is normalized to the catalogue maximum so the components are comparable.


### 7. Dimesionality Reduction

Dimensionality reduction is unnecessary because there are only nine hotel items.


Answer Here.

In [ ]:
print('No dimensionality reduction applied.')


No dimensionality-reduction technique is required for a nine-item catalogue.


Answer Here.

### 8. Data Splitting

In [ ]:
# Temporal holdout: last booking per eligible user becomes test target
eligible=hotels.groupby('userCode').size(); eligible_users=eligible[eligible>=2].index; ordered=hotels[hotels.userCode.isin(eligible_users)].sort_values(['userCode','date','travelCode']); test_idx=ordered.groupby('userCode').tail(1).index; test_events=ordered.loc[test_idx].copy(); train_events=ordered.drop(test_idx).copy(); print(train_events.shape,test_events.shape)


##### What data splitting ratio have you used and why? 

A temporal leave-last-one-out split is used. For every user with at least two bookings, the latest booking is held out as the evaluation target and all earlier bookings form the recommendation history.


### 9. Handling Imbalanced Dataset

Class imbalance is not directly applicable to recommendation ranking, although hotel popularity is uneven.


Answer Here.

In [ ]:
print(train_events['name'].value_counts(normalize=True))


No resampling is performed. Popularity differences are part of the real recommendation environment and are modeled explicitly as a baseline/fallback.


Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# Build temporal training interaction matrix and evaluation utilities
items=sorted(hotels['name'].unique()); item_to_idx={item:i for i,item in enumerate(items)}; users=sorted(train_events['userCode'].unique()); user_to_idx={u:i for i,u in enumerate(users)}
train_matrix=np.zeros((len(users),len(items)),dtype=float)
for row in train_events.itertuples(): train_matrix[user_to_idx[row.userCode],item_to_idx[row.name]] += 1
popularity=train_matrix.sum(axis=0); popularity=popularity/popularity.max()

def evaluate_ranker(score_function,k=3):
    hits=[]; reciprocal_ranks=[]
    for row in test_events.itertuples():
        if row.userCode not in user_to_idx: continue
        scores=score_function(row.userCode)
        top=np.argsort(-scores)[:k]
        target=item_to_idx[row.name]
        if target in top:
            rank=list(top).index(target)+1; hits.append(1); reciprocal_ranks.append(1/rank)
        else:
            hits.append(0); reciprocal_ranks.append(0)
    return {'HitRate@'+str(k):float(np.mean(hits)),'MRR@'+str(k):float(np.mean(reciprocal_ranks))}

popularity_results=evaluate_ranker(lambda user_code: popularity.copy(),k=3)
print(popularity_results)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Model 1 is a non-personalized popularity recommender. It ranks hotels by total historical booking volume and provides the cold-start baseline. The validated benchmark achieved **Hit Rate@3 ≈ 0.353** and **MRR@3 ≈ 0.222**.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
for k in [1,3,5]: print(k,evaluate_ranker(lambda u: popularity.copy(),k=k))


##### Which hyperparameter optimization technique have you used and why?

The popularity model has no learned hyperparameters; different recommendation list lengths K are compared.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Increasing K improves coverage but reduces recommendation focus. K=3 is used as the primary compact recommendation list.


### ML Model - 2

In [ ]:
# ML Model - 2: Personalized frequency recommender
def frequency_scores(user_code):
    history=train_matrix[user_to_idx[user_code]]
    normalized=history/(history.max() if history.max()>0 else 1)
    return 0.8*normalized + 0.2*popularity
frequency_results=evaluate_ranker(frequency_scores,k=3); print(frequency_results)


The personalized frequency recommender combines each user’s normalized hotel booking frequency with a small global popularity prior. It achieved the best validated result at **Hit Rate@3 ≈ 0.391** and **MRR@3 ≈ 0.236**.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
weights=[]
for personal_weight in [0.6,0.7,0.8,0.9,1.0]:
    def scorer(u,w=personal_weight):
        h=train_matrix[user_to_idx[u]]; h=h/(h.max() if h.max()>0 else 1); return w*h+(1-w)*popularity
    metrics=evaluate_ranker(scorer,k=3); weights.append({'personal_weight':personal_weight,**metrics})
display(pd.DataFrame(weights).sort_values('HitRate@3',ascending=False))


##### Which hyperparameter optimization technique have you used and why?

A small manual/grid search over the personal-history versus popularity weight is used because the ranking formula has one interpretable parameter.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

A strong personal-history weight with a small popularity prior performs best, confirming that repeat preference is useful in this dataset.


#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Hit Rate@K measures how often the held-out future hotel appears anywhere in the top-K recommendations. MRR@K additionally rewards placing the correct hotel nearer the top of the list. Both metrics map directly to recommendation usefulness.


### ML Model - 3

In [ ]:
# ML Model - 3: Hybrid similarity, history, price and popularity
binary_matrix=(train_matrix>0).astype(float); item_similarity=cosine_similarity(binary_matrix.T); item_prices=hotels.groupby('name')['price'].median().reindex(items).values

def hybrid_scores(user_code):
    history=train_matrix[user_to_idx[user_code]]
    normalized=history/(history.max() if history.max()>0 else 1)
    history_share=history/(history.sum() if history.sum()>0 else 1)
    collaborative=history_share @ item_similarity
    user_avg_price=train_events.loc[train_events.userCode==user_code,'price'].mean()
    price_pref=np.exp(-np.abs(item_prices-user_avg_price)/(user_avg_price+1e-9))
    return 0.45*collaborative + 0.25*normalized + 0.15*price_pref + 0.15*popularity
hybrid_results=evaluate_ranker(hybrid_scores,k=3)
results=pd.DataFrame([{'Model':'Popularity',**popularity_results},{'Model':'Personalized Frequency',**frequency_results},{'Model':'Hybrid',**hybrid_results}]); display(results)


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

The hybrid model adds item-item similarity and price preference. In the validated benchmark it achieved **Hit Rate@3 ≈ 0.366**, below the simpler personalized frequency model. The result is reasonable because users already interact with most of the nine-item catalogue, leaving little benefit for complex similarity modeling.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Compare hybrid behavior across K values
for k in [1,3,5]: print(k,evaluate_ranker(hybrid_scores,k=k))


##### Which hyperparameter optimization technique have you used and why?

The hybrid weights are kept interpretable rather than heavily optimized on the test set. Evaluation across multiple K values checks ranking behavior without overfitting the held-out interactions.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The hybrid model did not outperform personalized frequency on Hit Rate@3, so added complexity is not justified for the final deployed recommender.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Hit Rate@3 is the primary metric because the Streamlit app will show a short list of recommendations. MRR@3 is used as a secondary metric to reward correct items appearing closer to rank one.


### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The **Personalized Frequency Recommender with a popularity fallback** is selected because it achieved the best validated Hit Rate@3, is transparent, fast, and appropriate for the dense nine-hotel catalogue.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Explain a recommendation for one example user
example_user=users[0]; scores=frequency_scores(example_user); top_idx=np.argsort(-scores)[:3]; explanation=pd.DataFrame({'hotel':[items[i] for i in top_idx],'score':[scores[i] for i in top_idx],'past_bookings':[train_matrix[user_to_idx[example_user],i] for i in top_idx]}); display(explanation)


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save recommender state for Streamlit deployment
MODEL_DIR=Path('models'); MODEL_DIR.mkdir(exist_ok=True); recommender_state={'items':items,'user_to_idx':user_to_idx,'train_matrix':train_matrix,'popularity':popularity,'hotel_metadata':hotels.groupby('name').agg(place=('place','first'),price=('price','median'),bookings=('name','size')).reset_index(),'personal_weight':0.8}; MODEL_PATH=MODEL_DIR/'hotel_recommender.joblib'; joblib.dump(recommender_state,MODEL_PATH,compress=3); print(MODEL_PATH.resolve())


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
loaded=joblib.load(MODEL_PATH); print('Hotels:',loaded['items']); print('Stored users:',len(loaded['user_to_idx']))


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

The hotel recommendation workflow demonstrates that model complexity should match the structure of the data. The catalogue contains only nine hotels and user interactions are dense, so a transparent personalized-frequency recommender outperformed both a non-personalized popularity baseline and the tested multi-signal hybrid on Hit Rate@3. Temporal holdout evaluation ensured that recommendations were tested against a later booking rather than interactions used for training.

The final recommender combines strong user booking-history scores with a small popularity prior and falls back entirely to popularity for cold-start users. Its state is saved with Joblib and is ready to power the Streamlit travel recommendation application. The simple ranking logic also makes recommendation reasons easy to communicate to users and evaluators.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***